# #2 UMAP Embedding

## Purpose

Generate global and cluster UMAP embeddings

## Setup

In [1]:
import scanpy as sc
import cupy as cp
import rapids_singlecell as rsc
import rmm
import pandas as pd

from devmap.config import set_theme, get_paths
from devmap.utils import load_data

set_theme()
base_path, data_path, results_path = get_paths("embedding")

## GPU configuration

In [2]:
rmm.reinitialize(
    managed_memory=True,  # Allows oversubscription
    pool_allocator=False,  # default is False
    devices=0,  # GPU device IDs to register. By default registers only GPU 0.
)
cp.cuda.set_allocator(rmm.allocators.cupy.rmm_cupy_allocator)

## Load data

In [ ]:
tdata = load_data("log1p",scvi=True)

## Global UMAP embedding

In [ ]:
rsc.pp.neighbors(tdata, use_rep="X_scvi", n_neighbors=50)
rsc.tl.umap(tdata)
sc.pl.umap(tdata, color = "cell_type", legend_loc="on data", legend_fontsize=6)

save embedding

In [ ]:
umap = pd.DataFrame(tdata.obsm["X_umap"], index=tdata.obs_names, columns=["UMAP1", "UMAP2"])
umap.to_csv(base_path / "data" / "umap.csv")

## Cluster UMAP embedding

In [ ]:
cluster_umaps = []
for cluster in tdata.obs["cluster"].unique():
    cluster_tdata = tdata[tdata.obs.query("cluster == @cluster").index,:].copy()
    rsc.pp.neighbors(cluster_tdata, use_rep="X_scvi", n_neighbors=50)
    rsc.tl.umap(cluster_tdata)
    sc.pl.umap(cluster_tdata, color = "cell_subtype", legend_loc="on data", legend_fontsize=6, title=cluster)
    cluster_tdata.write_h5ad(f"/lab/wcolgan_scratch/{cluster.replace(' ', '_')}.h5ad")
    umap = pd.DataFrame(cluster_tdata.obsm["X_umap"], index=cluster_tdata.obs_names, columns=["UMAP1", "UMAP2"])
    cluster_umaps.append(umap)

save embeddings

In [ ]:
cluster_umaps = pd.concat(cluster_umaps)
cluster_umaps.to_csv(results_path / 'cluster_umaps.csv')